# The sickle-cell variant in HBB

Sickle-cell disease comes from a single-letter change in *HBB*, the gene for the beta chain of hemoglobin on human chromosome 11. This notebook loads 6,000 bases of the human genome around that gene, adds the gene's annotation, and applies the sickle-cell variant (rs334) on its own branch. Then it checks what the change does to the protein and writes the result back out.

The data sits beside this notebook and is described in [hbb_data_sources.md](hbb_data_sources.md), which gives the exact sources, positions, access date and usage terms:

| File | Content | Source |
|------|---------|--------|
| `hbb_reference.fa` | human genome (GRCh38), chromosome 11, positions 5,224,001 to 5,230,000 | Ensembl |
| `hbb_genes.gff3` | the *HBB* gene, its main transcript and exons | Ensembl, release 116 |
| `hbb_sickle_cell.vcf` | variant rs334, chromosome 11 position 5,227,002, T to A | Ensembl and ClinVar (VCV000015333) |

*HBB* is read from the reverse strand of the chromosome, so the sequence you see in the graph is the complement of the gene's own letters. The notebook points this out where it matters.

## Load the reference

A Gen repository keeps sequences, annotations and every change you make to them. The reference sequence goes in first, under the sample name `reference`.

In [1]:
import pathlib
import tempfile

import gen

EXAMPLES_DIR = pathlib.Path(__file__).parent if "__file__" in dir() else pathlib.Path(".").resolve()

repo = gen.Repository(tempfile.mkdtemp(prefix="gen-hbb-"))
reference = repo.import_fasta(str(EXAMPLES_DIR / "hbb_reference.fa"), sample="reference")

reference_graph = reference[0]
print(f"{reference_graph.name}: {len(str(reference_graph.all_sequences()[0]))} bases")

chr11_5224001_5230000: 6000 bases


## Add the gene annotation

The GFF3 file marks where the gene, its transcript, its three exons and its coding sequence lie. Once imported, the features appear in the graph and can be listed from Python.

In [2]:
repo.import_annotations(str(EXAMPLES_DIR / "hbb_genes.gff3"), name="HBB")

for annotation in sorted(reference_graph.annotations, key=lambda feature: feature.name):
    print(f"{annotation.name:22s} {len(annotation):5d} bases")

HBB                     3932 bases
HBB coding sequence      444 bases
HBB exon 1               142 bases
HBB exon 2               223 bases
HBB exon 3               263 bases
HBB-201                 1608 bases


## Find the start of the gene

The protein begins with the codon ATG. Because *HBB* is on the reverse strand, that codon appears here as its reverse complement, CAT, at the right-hand end of the first exon's coding part. Searching for the first stretch of the gene's coding sequence (ATG GTG CAT CTG ACT CCT GAG GAG) finds it on the reverse strand, in the first exon. Position 3,002, where the variant will go, lies a short way to the left of it in the plot. The exon labels sit under the sequence.

In [3]:
start_hit = reference_graph.search("ATGGTGCATCTGACTCCTGAGGAG")[0]
print(f"Found on strand {start_hit.strand}: {start_hit.sequence}")

reference_graph.plot(rows=24, detail="full", go_to=reference_graph.locus(2985, 3025))

Found on strand -: ATGGTGCATCTGACTCCTGAGGAG


                   chr11_5224001_5230000









     ↓
gcagtAacggcagacttctcctcaggagtcagatgcaccatggtgtctgtttgaggttgc
═════════════════════════════════════════════════════════hbb
═══════════════hbb exon 1═══════════════════════════════════
hbb-201═════════════════════════════════════════════════════
hbb coding sequence═════════════════════]






|...................|...................|...................
2,980             3,000               3,020

## Apply the sickle-cell variant on its own branch

Branches work as they do in Git. The reference stays on `main`, and the variant goes on a new branch called `sickle-cell`, so the normal sequence is never overwritten.

The VCF file holds one line: rs334 at position 3,002 of this sequence (chromosome 11 position 5,227,002), where the reference has T and the variant has A. On the gene's own strand this is codon 7 changing from GAG (glutamic acid) to GTG (valine). Two example people carry it: `sickle_cell` has two copies and `sickle_trait` has one.

In [4]:
repo.checkout("sickle-cell", create=True)
people = repo.update_with_vcf(str(EXAMPLES_DIR / "hbb_sickle_cell.vcf"), reference="reference")

Checking out branch sickle-cell


In [5]:
print((EXAMPLES_DIR / "hbb_sickle_cell.vcf").read_text().splitlines()[-1])
print()

graphs = {person.name: person[0] for person in sorted(people, key=lambda person: person.name)}
for name, graph in graphs.items():
    print(f"{name}: {len(graph.all_sequences())} distinct sequence(s) across the region")

chr11_5224001_5230000	3002	rs334	T	A	.	PASS	.	GT	1/1	0/1

sickle_cell: 1 distinct sequence(s) across the region
sickle_trait: 2 distinct sequence(s) across the region


## See the variant in the gene

A person with one copy of the variant, `sickle_trait`, has both versions of the sequence, so the graph forks at the variant: one route carries the normal T and the other the sickle A. The labels under the sequence show that the change falls inside the gene's first exon, in its coding sequence.

In [6]:
trait_graph = graphs["sickle_trait"]
variant_hit = trait_graph.search("CTCCTCAGG")[0]
trait_graph.plot(rows=24, detail="full", go_to=variant_hit)

                   chr11_5224001_5230000








          ╭─a─╮
     ↓    │   │
gacttCtcc─┴─t─┴─caggagtcagatgcaccatggtgtctgtttgaggttgctagtga
hbb══════   ═   ═════════════════════════════════════════hbb
═════════   ═   ═════════════════════════════hbb exon 1═════
hbb-201══   ═   ═══════════════════════════════hbb-201══════
═════════   ═   ══════════════════]






........|.........................|...................|.....
      3,000                     3,020               3,040

## Translate the gene to protein

`translate_annotation` turns a feature into a protein graph. It reads the DNA exactly as written, so it needs a stretch of sequence without introns. The first exon holds the start of the protein (the first 30 amino acids), so we cut that exon out of each person's graph and translate it, reading the reverse strand.

The protein graphs of the three people show the difference: `sickle_cell` has valine at position 7, and `sickle_trait` has both residues, glutamic acid on one route and valine on the other.

In [7]:
FIRST_CODING_EXON = (2929, 3021)

proteins = {}
for name, graph in graphs.items():
    exon = graph.subgraph(f"{name}_first_exon", *FIRST_CODING_EXON)
    proteins[name] = exon.translate_annotation(strand="reverse")

Derive subgraph succeeded.
Derive subgraph succeeded.


In [8]:
protein_sequences = {
    name: [str(sequence) for sequence in protein.all_sequences()] for name, protein in proteins.items()
}

for name, sequences in protein_sequences.items():
    for sequence in sequences:
        print(f"{name:13s} {sequence}  residue 7: {sequence[6]}")

proteins["sickle_trait"].plot(rows=12, detail="full")

sickle_cell   MVHLTPVEKSAVTALWGKVNVDEVGGEALG  residue 7: V
sickle_trait  MVHLTPEEKSAVTALWGKVNVDEVGGEALG  residue 7: E
sickle_trait  MVHLTPVEKSAVTALWGKVNVDEVGGEALG  residue 7: V


              chr11_5224001_5230000 (protein)




            ╭─v─╮
  ╟──mvhltp─┤   ├─eksavtalwgkvnvdevggealg──╢
            ╰─e─╯




## Compare the normal and sickle alleles

Switch back to `main` to see the normal allele. Each branch holds its own samples. Python has no single call that diffs two branches, so compare what each one holds, here the protein each allele encodes.

In [9]:
repo.checkout("main")
normal_exon = repo.get_sample("reference")[0].subgraph("reference_first_exon", *FIRST_CODING_EXON)
normal_protein = normal_exon.translate_annotation(strand="reverse")

Checking out branch main
Derive subgraph succeeded.


In [10]:
normal_sequence = str(normal_protein.all_sequences()[0])

print("samples on main:", sorted(sample.name for sample in repo.samples))
print()
sickle_sequence = protein_sequences["sickle_cell"][0]
print("normal protein  ", normal_sequence)
print("sickle protein  ", sickle_sequence)

differences = [
    (position + 1, normal, sickle)
    for position, (normal, sickle) in enumerate(zip(normal_sequence, sickle_sequence))
    if normal != sickle
]
print("differences     ", differences)

samples on main: ['reference', 'reference_first_exon']

normal protein   MVHLTPEEKSAVTALWGKVNVDEVGGEALG
sickle protein   MVHLTPVEKSAVTALWGKVNVDEVGGEALG
differences      [(7, 'E', 'V')]


## Export the variant sample

`export_fasta` writes the sample's sequences. By default it follows the reference path, so pass `all_sequences=True` to write every route through the graph, which here is the one sequence that carries the variant. `export_gfa` writes the whole graph, with both alleles for the carrier. Both come from the `sickle-cell` branch.

In [11]:
repo.checkout("sickle-cell")

output_dir = pathlib.Path(tempfile.mkdtemp(prefix="gen-hbb-export-"))
graphs["sickle_cell"].export_fasta(str(output_dir / "sickle_cell.fa"), all_sequences=True)
graphs["sickle_trait"].export_gfa(str(output_dir / "sickle_trait.gfa"))

Checking out branch sickle-cell


In [12]:

exported = "".join(line for line in (output_dir / "sickle_cell.fa").read_text().splitlines() if not line.startswith(">"))
reference_bases = "".join(line for line in (EXAMPLES_DIR / "hbb_reference.fa").read_text().splitlines() if not line.startswith(">"))
print(f"base 3002 in the reference: {reference_bases[3001]}   in the export: {exported[3001]}")
print(f"{sum(a != b for a, b in zip(reference_bases, exported))} base differs, {len(exported)} bases exported")

segments = [line.split("	")[2] for line in (output_dir / "sickle_trait.gfa").read_text().splitlines() if line.startswith("S")]
print("GFA segments:", sorted(segment[:12] for segment in segments))

base 3002 in the reference: T   in the export: A
1 base differs, 6000 bases exported
GFA segments: ['A', 'CAGGAGTCAGAT', 'T', 'TAAATGGCAGGA']
